# 🔥 Advanced · RLHF with PPO

> 🔥 **Advanced / heavy lab.** Fine-tune an LLM with reinforcement learning against a reward model — the classic RLHF recipe.
>
> **Runtime → Change runtime type → T4 GPU required.** Est. **20–40 min** including downloads. Built on the official **[TRL PPOTrainer](https://huggingface.co/docs/trl/ppo_trainer)** and authored to its documented recipe — **not pre-executed here** (needs a GPU + large downloads). If a step fails, see *Troubleshooting* at the bottom; pin versions as noted.

The scaled-up cousin of the REINFORCE lab (track AG): here the policy is a language model and the reward comes from a learned model. DPO (other lab) is the RL-free alternative.

## Compute · storage · time

| Resource | Demo (this notebook, free Colab T4) | Full / production run |
|---|---|---|
| **GPU** | T4 16 GB — GPT-2 sentiment demo | multi-GPU 40–80 GB (policy + ref + reward + value for 7B) |
| **Storage** | small | 3–4 model copies + rollout logs; 100 GB+ |
| **Time** | ~ tens of minutes | 7B RLHF ~ days on 8× A100 |

**Full pipeline (scale-up):** train a reward model, then `accelerate launch ppo.py …` on full prompts (DPO is a cheaper alternative).

> Rough estimates — real numbers depend on hardware, data size and library versions.

In [ ]:
!nvidia-smi -L
!pip install -q "trl<0.12" transformers datasets

## 1 · Policy (with a value head) + a reward model
Classic demo: steer GPT-2 toward **positive sentiment**, scored by a sentiment classifier.

In [ ]:
import torch
from trl import PPOConfig, PPOTrainer, AutoModelForCausalLMWithValueHead
from transformers import AutoTokenizer, pipeline
base = "lvwerra/gpt2-imdb"
tok = AutoTokenizer.from_pretrained(base); tok.pad_token = tok.eos_token
model = AutoModelForCausalLMWithValueHead.from_pretrained(base)
ref = AutoModelForCausalLMWithValueHead.from_pretrained(base)
reward_fn = pipeline("sentiment-analysis", model="lvwerra/distilbert-imdb", device=0 if torch.cuda.is_available() else -1)

## 2 · PPO loop — generate, score, update

In [ ]:
from datasets import load_dataset
ds = load_dataset("imdb", split="train").shuffle(seed=0).select(range(200))
ppo = PPOTrainer(PPOConfig(batch_size=16, mini_batch_size=4, learning_rate=1.4e-5), model, ref, tok)
gen_kw = dict(max_new_tokens=24, do_sample=True, top_k=0, top_p=1.0, pad_token_id=tok.eos_token_id)
for batch in [ds[i:i+16] for i in range(0, 64, 16)]:
    queries = [tok(t[:40], return_tensors="pt").input_ids[0] for t in batch["text"]]
    responses = [ppo.generate(q, **gen_kw).squeeze()[len(q):] for q in queries]
    texts = [tok.decode(r) for r in responses]
    rewards = [torch.tensor(s["score"] if s["label"] == "POSITIVE" else 1 - s["score"]) for s in reward_fn(texts)]
    stats = ppo.step(queries, responses, rewards)
    print("mean reward:", round(float(torch.stack(rewards).mean()), 3))

## Save & persist your results
This pipeline writes its checkpoints, metrics/logs and outputs to the run/output directory shown above (e.g. `output/`, `outputs/`, `logdir/`, `experiments/`, or the trainer's `output_dir`). **Colab sessions are ephemeral** — to keep them, either mount Drive and copy the folder (`from google.colab import drive; drive.mount('/content/drive')`) or zip + download it (`import shutil; shutil.make_archive('run','zip','OUTPUT_DIR')` then `from google.colab import files; files.download('run.zip')`). The 🤗 Trainer labs also support `trainer.push_to_hub()`. To publish any output folder as a **model repo** (then group repos into a **Collection** on your profile): `from huggingface_hub import HfApi; HfApi().upload_folder(folder_path="OUTPUT_DIR", repo_id="<you>/ropedia-<lab>")`.

## How this links to tracks A–D
RL fine-tuning generalises:
- **A · Human** align a motion generator (MDM) to preferred motions · **D · Scene / world** RL alignment for agents and world-model planning.

## Troubleshooting & next steps
- **TRL version**: the PPO API changed across releases; pin (`trl<0.12`) to match this script, or follow the example for your installed version.
- Train a **reward model** from preference data first, then PPO against it → full RLHF.
- Compare with **DPO** (simpler, no reward model / rollouts) — the other LM lab.